In [1]:
pip install "flask<2.3,>=2.2"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.8/101.8 kB 1.6 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.2/225.2 kB 2.4 MB/s eta 0:00:00a 0:00:01
Note: you may need to restart the kernel to use updated packages.


In [3]:
!pip install python-dotenv

In [2]:
pip install pyspark boto3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.5/200.5 kB 2.4 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.6/14.6 MB 26.4 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 3.8 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [3]:
import os
from pyspark.sql import SparkSession

# Spark com libs do S3
spark = (
    SparkSession.builder
    .appName("ETL-IPCA-Silver-Gold")
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262")
    .getOrCreate()
)
from pyspark.sql.functions import col, from_unixtime, avg, count
from dotenv import load_dotenv

In [4]:
# Carrega credenciais
load_dotenv(".env_kafka_connect")

aws_access_key = os.getenv("AWS_ACCESS_KEY_ID")
aws_secret_key = os.getenv("AWS_SECRET_ACCESS_KEY")
aws_region = os.getenv("AWS_REGION", "us-east-1")

In [5]:
# Inicializar a Spark Session com Configuração S3

# Caminho local dos arquivos JAR no mesmo diretório do notebook
# Obter caminho absoluto dos JARs
current_dir = os.getcwd()
hadoop_aws_jar = os.path.join(current_dir, "hadoop-aws-3.3.4.jar")
aws_sdk_jar = os.path.join(current_dir, "aws-java-sdk-bundle-1.12.262.jar")

jars_path = f"{hadoop_aws_jar},{aws_sdk_jar}"

spark = SparkSession.builder \
    .appName("ETL Pipeline - S3 Integration") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.jars", jars_path) \
    .getOrCreate()

# Configuração dinâmica das credenciais AWS no Spark
spark._jsc.hadoopConfiguration().set("fs.s3a.access.key", aws_access_key)
spark._jsc.hadoopConfiguration().set("fs.s3a.secret.key", aws_secret_key)
spark._jsc.hadoopConfiguration().set("fs.s3a.endpoint", f"s3.{aws_region}.amazonaws.com")

# Configuração do acesso ao S3
spark._jsc.hadoopConfiguration().set("fs.s3a.access.key", aws_access_key)
spark._jsc.hadoopConfiguration().set("fs.s3a.secret.key", aws_secret_key)
spark._jsc.hadoopConfiguration().set("fs.s3a.endpoint", f"s3.{aws_region}.amazonaws.com")
spark._jsc.hadoopConfiguration().set("fs.s3a.connection.ssl.enabled", "true")
spark._jsc.hadoopConfiguration().set("fs.s3a.path.style.access", "true")

In [6]:
# Ler os Dados Brutos - Camada Bronze
bronze_path = "s3a://my-bucket-ek-1/raw-data/ipca/kafka/"

# Ler os dados do S3 e testar a conexão
try:
    df_bronze = spark.read.json(bronze_path)
    print("Leitura bem-sucedida!")
    df_bronze.show()
except Exception as e:
    print(f"Erro ao acessar o S3: {e}")


Leitura bem-sucedida!
+-----------+-------------+---------------+-----------+-------------+------------+----+----------+-------------+---------+
|CompraManha|    Data_Base|Data_Vencimento|PUBaseManha|PUCompraManha|PUVendaManha|Tipo|VendaManha|    dt_update|partition|
+-----------+-------------+---------------+-----------+-------------+------------+----+----------+-------------+---------+
|      10.02|1771545600000|  1786752000000|    4393.66|      4399.75|     4393.66|IPCA|     10.14|1772036153509|        0|
|       10.2|1770595200000|  1786752000000|    4371.93|      4376.51|     4371.93|IPCA|     10.32|1772036153510|        0|
|      10.21|1769990400000|  1786752000000|    4359.96|      4364.63|     4359.96|IPCA|     10.33|1772036153516|        0|
|       7.59|1769990400000|  1976140800000|    2836.45|      2858.19|     2836.45|IPCA|      7.71|1772036153516|        0|
|       7.01|1769731200000|  2378419200000|    1227.06|      1254.44|     1227.06|IPCA|      7.13|1772036153518|     

In [7]:
# Tratamento dos Dados - Camada Silver
# Remover duplicações e converter timestamps para datas legíveis

df_silver = df_bronze.dropDuplicates()

# Tratar timestamps e converter para formato legível
df_silver = df_silver.withColumn("Data_Vencimento", from_unixtime(col("Data_Vencimento") / 1000, "yyyy-MM-dd")) \
                     .withColumn("Data_Base", from_unixtime(col("Data_Base") / 1000, "yyyy-MM-dd")) \
                     .withColumn("dt_update", from_unixtime(col("dt_update") / 1000, "yyyy-MM-dd HH:mm:ss"))

# Tratar valores nulos
df_silver = df_silver.fillna({
    "PUCompraManha": 0,
    "PUVendaManha": 0,
    "PUBaseManha": 0
})

# Visualizar os dados transformados
print("Dados Transformados (Silver):")
df_silver.show(truncate=False)

# Salvar os dados limpos no S3 em formato Parquet
silver_path = "s3a://my-bucket-ek-1/processed-data/ipca/silver/"
df_silver.write.mode("overwrite").parquet(silver_path)

Dados Transformados (Silver):
+-----------+----------+---------------+-----------+-------------+------------+----+----------+-------------------+---------+
|CompraManha|Data_Base |Data_Vencimento|PUBaseManha|PUCompraManha|PUVendaManha|Tipo|VendaManha|dt_update          |partition|
+-----------+----------+---------------+-----------+-------------+------------+----+----------+-------------------+---------+
|7.59       |2026-01-16|2035-05-15     |2305.57    |2330.98      |2305.57     |IPCA|7.71      |2026-02-25 16:15:53|0        |
|10.29      |2025-12-22|2026-08-15     |4292.35    |4297.62      |4292.35     |IPCA|10.41     |2026-02-25 16:15:53|0        |
|10.04      |2025-12-02|2026-08-15     |4268.58    |4273.74      |4268.58     |IPCA|10.16     |2026-02-25 16:15:53|0        |
|8.05       |2025-10-17|2029-05-15     |3453.17    |3468.52      |3453.17     |IPCA|8.17      |2026-02-25 16:15:53|0        |
|6.93       |2026-01-02|2045-05-15     |1234.44    |1262.11      |1234.44     |IPCA|7.05

In [8]:
# Agregação e Métricas - Camada Gold
# Calcular métricas agregadas
df_gold = df_silver.groupBy("Tipo").agg(
    avg("PUCompraManha").alias("Media_PUCompraManha"),
    avg("PUVendaManha").alias("Media_PUVendaManha"),
    count("*").alias("Total_Registros")
)

# Visualizar as métricas agregadas
print("Dados Agregados (Gold):")
df_gold.show(truncate=False)

# Salvar os dados agregados no S3 em formato Parquet
gold_path = "s3a://my-bucket-ek-1/analytics/ipca/gold/"
df_gold.write.mode("overwrite").parquet(gold_path)

Dados Agregados (Gold):
+----+-------------------+------------------+---------------+
|Tipo|Media_PUCompraManha|Media_PUVendaManha|Total_Registros|
+----+-------------------+------------------+---------------+
|IPCA|1810.413387096772  |1795.6990008960572|17856          |
+----+-------------------+------------------+---------------+



In [9]:
# Encerrar a Spark Session
spark.stop()